### Objetivo y alcance de la versión 4

Demostración académica de mantenimiento de un pipeline tabular. El objetivo y los
cambios de distribución son sintéticos; los siete vitales proceden de PhysioNet.
Las cifras y figuras se calculan, nunca se prescriben. Ejecutar de arriba abajo.

La versión 4 conserva de v3 **todo el agente y el entrenamiento**: episodios
(500–1000), Adam 5e-4 → 5e-5, gamma 0,99, Double DQN, pérdida Huber, epsilon,
replay, red target, selección de checkpoint y recompensa con costo normalizado.

Lo único que cambia es **la forma del cambio estadístico**. En v2/v3 cada etapa
aparecía de golpe, lo que producía caídas de AUC de un día seguidas de
recuperación. En v4 la severidad **crece de forma gradual cada día**:

- **Covariate:** el desplazamiento de medición aumenta una fracción de IQR por día,
  sin tope, durante todo el resto del episodio.
- **Concept:** la relación features–etiqueta se desplaza continuamente entre las
  relaciones de v2/v3 (HR invertido → HR y Resp → Resp).
- **Combined:** ambas rampas; la de concepto empieza dos días después.

La degradación diaria es pequeña al principio y se acumula. Una actualización no
detiene el cambio: el agente debe decidir **cuándo** la pérdida acumulada justifica
el costo de actuar. Las velocidades se calibran con B-train antes de entrenar.

### Bibliotecas, rutas y parámetros reproducibles

Se fijan semillas en 42, la lista de vitales, los escenarios y los parámetros
de v3 sin cambios (episodios, tasa de aprendizaje, `COST_LAMBDA`, epsilon) y los
nuevos de v4: rejillas de velocidad de cambio, retraso del concepto y factores de
velocidad usados durante el entrenamiento.

In [ ]:
import os, json, time, copy, random, traceback, hashlib, platform
from pathlib import Path
from collections import deque
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import xgboost, sklearn
import torch
from torch import nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 42
VITALS = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp']
WEIGHTS = np.array([1.2, -1.1, .3, -.2, -.7, .2, .8], dtype=np.float32)
SCENARIOS = ('stable', 'covariate', 'concept', 'combined')
ACTIONS = ('A1', 'A2', 'A3', 'A4')
SMOKE = os.environ.get('DARL_DUMMY_SMOKE') == '1'
DAYS = 4 if SMOKE else 16
EPISODES = 2 if SMOKE else 200
# v4: rampas graduales; inicio fijo en validación/prueba.
ONSET = 2 if SMOKE else 5
COV_RATE_GRID = (.25, .35, .5)        # IQR por día
CONCEPT_RATE_GRID = (.1, .15, .2)     # unidades de trayectoria conceptual por día
CONCEPT_DELAY = 2                     # días de retraso del concepto en combined
CONCEPT_MAX = 3.                      # tres anclas tras la relación original
SCALE_GAIN = .4                       # escala máxima 1,4 a partir de 3 IQR
CALIBRATION_DAYS = 6                  # día de verificación de degradación fuerte
TRAIN_RATE_FACTORS = (.75, 1., 1.25)
# Anclas de concepto: signos aplicados a WEIGHTS (HR; HR+Resp; Resp).
CONCEPT_SIGNS = np.array([[-1, 1, 1, 1, 1, 1, 1],
                          [-1, 1, 1, 1, 1, 1, -1],
                          [ 1, 1, 1, 1, 1, 1, -1]], dtype=np.float32)
# v2 (C): costo = COST_LAMBDA x segundos / mediana de segundos de A4.
COST_LAMBDA = .05
# v2 (D): seis de siete episodios con cambio; epsilon decae por episodio.
TRAIN_CYCLE = ('stable', 'covariate', 'concept', 'combined', 'covariate', 'concept', 'combined')
EPS_DECAY, EPS_MIN = .99, .05
# v3: entrenamiento más largo y más lento; selección de checkpoint suavizada.
MIN_EPISODES, MAX_EPISODES = (4, 4) if SMOKE else (500, 1000)
LR_START, LR_END = 5e-4, 5e-5
SELECT_WINDOW = 3
Q_TIE = .005
OUT = Path(os.environ.get('DARL_DUMMY_OUTPUT', '/kaggle/working' if Path('/kaggle/input').exists() else 'kaggle/dqn-dummy/v4/outputs'))
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'figures').mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(2)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
progress = {'status': 'running', 'stage': 'setup', 'version': 'v4', 'seed': SEED, 'device': str(DEVICE), 'smoke': SMOKE,
            'design': {'drift': 'gradual', 'onset': ONSET, 'cov_rate_grid': COV_RATE_GRID,
                       'concept_rate_grid': CONCEPT_RATE_GRID, 'concept_delay': CONCEPT_DELAY,
                       'train_rate_factors': TRAIN_RATE_FACTORS, 'cost_lambda': COST_LAMBDA,
                       'train_cycle': list(TRAIN_CYCLE), 'eps_decay_per_episode': EPS_DECAY, 'eps_min': EPS_MIN,
                       'episodes': [MIN_EPISODES, MAX_EPISODES], 'lr': [LR_START, LR_END], 'double_dqn': True,
                       'loss': 'huber', 'select_window': SELECT_WINDOW, 'q_tie': Q_TIE},
            'versions': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                         'sklearn': sklearn.__version__, 'xgboost': xgboost.__version__, 'torch': torch.__version__}}
def checkpoint(stage, **extra):
    """Persist a compact progress record after each costly stage."""
    progress.update(stage=stage, **extra)
    def clean(value):
        if isinstance(value, dict):return {str(k):clean(v) for k,v in value.items()}
        if isinstance(value, (list,tuple)):return [clean(v) for v in value]
        if isinstance(value,(float,np.floating)) and not np.isfinite(value):return None
        if isinstance(value,np.generic):return value.item()
        return value
    (OUT/'run_summary.json').write_text(json.dumps(clean(progress), indent=2, default=str, allow_nan=False), encoding='utf-8')
checkpoint('setup')

### Cargar columnas autorizadas y verificar llaves temporales

Igual que en v1: solo identificador, clínica, `ICULOS`, `SepsisLabel` y siete
vitales. Se aplica forward fill causal dentro de cada paciente.

In [ ]:
def find_parquet():
    """Resolve the one named Kaggle dataset or an explicit local test path."""
    custom = os.environ.get('DARL_DUMMY_PARQUET')
    candidates = [Path(custom)] if custom else list(Path('/kaggle/input').rglob('physionet.parquet'))
    candidates += [Path('data/processed/physionet/physionet.parquet')]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    mounts=sorted(p.name for p in Path('/kaggle/input').iterdir()) if Path('/kaggle/input').is_dir() else []
    raise FileNotFoundError(f'physionet.parquet unavailable; mounted inputs={mounts}')

DATA_PATH = find_parquet()
frame = pd.read_parquet(DATA_PATH, columns=['patient_id','source_set','ICULOS','SepsisLabel',*VITALS])
assert not frame.duplicated(['patient_id','ICULOS']).any()
assert set(frame.source_set.unique()) == {'a','b'}
frame = frame.sort_values(['patient_id','ICULOS']).reset_index(drop=True)
assert frame.groupby('patient_id').ICULOS.apply(lambda v: v.is_monotonic_increasing).all()
# ffill causal, sin mezclar pacientes; conservar los NaN originales para la inyección
frame[VITALS] = frame.groupby('patient_id', sort=False)[VITALS].ffill()
checkpoint('loaded', n_rows=len(frame), n_patients=int(frame.patient_id.nunique()))
print('Filas:', len(frame), '| pacientes:', frame.patient_id.nunique())

### Particiones por paciente, etiqueta artificial y coordenadas para el concepto

Particiones y `y_dummy` se construyen como en v1–v3. Para que la relación
features–etiqueta pueda moverse de forma continua se guardan, por fila, las
coordenadas robustas `z` usadas por el score (antes de cualquier cambio de
medición) y el bit fijo de inversión del 5%.

In [ ]:
patient_labels = frame.groupby(['source_set','patient_id'], sort=False).SepsisLabel.max().reset_index()
def split_ids(site, fractions):
    """Make deterministic patient-disjoint stratified partitions."""
    p = patient_labels.loc[patient_labels.source_set.eq(site)]
    ids, held = train_test_split(p.patient_id.to_numpy(), test_size=1-fractions[0], stratify=p.SepsisLabel, random_state=SEED)
    labels = p.set_index('patient_id').loc[held, 'SepsisLabel'].to_numpy()
    val, test = train_test_split(held, test_size=fractions[2]/(fractions[1]+fractions[2]), stratify=labels, random_state=SEED)
    return ids, val, test

a_train_ids, a_val_ids, a_test_ids = split_ids('a', (.70,.15,.15))
b_train_ids, b_val_ids, b_test_ids = split_ids('b', (.60,.20,.20))
all_parts = [set(x) for x in (a_train_ids,a_val_ids,a_test_ids,b_train_ids,b_val_ids,b_test_ids)]
assert all(not (all_parts[i] & all_parts[j]) for i in range(6) for j in range(i+1,6))

def roles(ids):
    """Partition B patients into update and held-out evaluation roles."""
    p = patient_labels.set_index('patient_id').loc[ids]
    u, e = train_test_split(ids, test_size=.5, stratify=p.SepsisLabel, random_state=SEED)
    assert not set(u) & set(e)
    return sorted(u), sorted(e)

B_PARTS = {'train':roles(b_train_ids), 'val':roles(b_val_ids), 'test':roles(b_test_ids)}
a_train = frame.loc[frame.patient_id.isin(a_train_ids)]
REF_MED = a_train[VITALS].median().to_numpy(dtype=np.float32)
REF_IQR = (a_train[VITALS].quantile(.75)-a_train[VITALS].quantile(.25)).clip(lower=1e-6).to_numpy(dtype=np.float32)
def canonical_scores(rows, weights=WEIGHTS):
    """Compute a synthetic target score on causal vitals before measurement change."""
    raw = rows[VITALS].to_numpy(dtype=np.float32)
    raw = np.where(np.isfinite(raw), raw, REF_MED)
    return (np.clip((raw-REF_MED)/REF_IQR, -4, 4)*weights).sum(axis=1)
THRESHOLD = float(np.median(canonical_scores(a_train)))
scores = canonical_scores(frame)
# Random flip is stable for a row regardless of replay order or partition.
def flip_bit(pid, hour):
    digest = hashlib.blake2b(f'{SEED}:{pid}:{hour}'.encode(), digest_size=8).digest()
    return int.from_bytes(digest, 'big') / 2**64 < .05
flips = np.fromiter((flip_bit(p,h) for p,h in zip(frame.patient_id,frame.ICULOS)), dtype=bool, count=len(frame))
frame['y_dummy'] = np.logical_xor(scores > THRESHOLD, flips).astype(np.int8)
assert frame.y_dummy.nunique() == 2
Z_COLS = [f'z_{v}' for v in VITALS]
def canonical_z(rows):
    """Return clipped robust coordinates of the synthetic target score."""
    raw = rows[VITALS].to_numpy(dtype=np.float32)
    raw = np.where(np.isfinite(raw), raw, REF_MED)
    return np.clip((raw-REF_MED)/REF_IQR, -4, 4).astype(np.float32)
frame[Z_COLS] = canonical_z(frame)
frame['flip'] = flips
assert np.allclose(frame[Z_COLS].to_numpy() @ WEIGHTS, scores, atol=1e-3)
ANCHORS = np.vstack([WEIGHTS, *(WEIGHTS*signs for signs in CONCEPT_SIGNS)]).astype(np.float32)
A_TRAIN_Z = frame.loc[frame.patient_id.isin(a_train_ids), Z_COLS].to_numpy()[::5]
_threshold_cache = {}
def concept_weights(level):
    """Interpolate weights along the anchor path; level 0 is the original relation."""
    level = float(np.clip(level, 0, CONCEPT_MAX))
    k = min(int(level), len(ANCHORS)-2)
    f = level-k
    return ANCHORS[k]*(1-f)+ANCHORS[k+1]*f
def concept_threshold(level):
    """Median A-train score for the interpolated relation (cached)."""
    key = round(float(level), 6)
    if key not in _threshold_cache:
        _threshold_cache[key] = float(np.median(A_TRAIN_Z @ concept_weights(level)))
    return _threshold_cache[key]
assert abs(concept_threshold(0)-THRESHOLD) < .05
concept_agreement = []
for level in (1., 2., 3.):
    moved = np.logical_xor(frame[Z_COLS].to_numpy() @ concept_weights(level) > concept_threshold(level), flips)
    concept_agreement.append(float((moved == frame.y_dummy.astype(bool)).mean()))
CONCEPT_THRESHOLDS = [concept_threshold(level) for level in (1., 2., 3.)]
a_train = frame.loc[frame.patient_id.isin(a_train_ids)]
checkpoint('partitioned', partitions={'a_train':len(a_train_ids),'a_val':len(a_val_ids),'a_test':len(a_test_ids),
                            'b_train':len(b_train_ids),'b_val':len(b_val_ids),'b_test':len(b_test_ids)},
           dummy_prevalence=float(frame.y_dummy.mean()), target_threshold=THRESHOLD,
           concept_thresholds=CONCEPT_THRESHOLDS, concept_agreement_with_dummy=concept_agreement)
print('Particiones disjuntas; prevalencia dummy:', round(frame.y_dummy.mean(),3),
      '| acuerdo y_dummy vs anclas 1-3:', np.round(concept_agreement,3))

### Replay de 200 camas con recambio horario y separación de roles

Sin cambios respecto a v1: 100 camas de actualización y 100 de evaluación; un
paciente agotado se reemplaza en la hora siguiente y nunca se reutiliza.

In [ ]:
PATIENT_LENGTHS = frame.groupby('patient_id',sort=False).size().to_dict()
PATIENT_TABLE = frame.assign(position=frame.groupby('patient_id',sort=False).cumcount()).set_index(['patient_id','position'])
def make_replay(parts, days=DAYS, seed=SEED):
    """Replay observed rows once per patient; replace exhausted beds next hour."""
    rng = np.random.default_rng(seed)
    selected = []
    for role, ids in zip(('update','eval'), parts):
        queue = list(rng.permutation(ids))
        beds = [(queue.pop(),0) for _ in range(100)]
        for hour in range(days*24):
            for bed in range(100):
                pid,pos = beds[bed]
                selected.append((pid,pos,role,bed+(0 if role=='update' else 100),hour))
                pos += 1
                if pos == PATIENT_LENGTHS[pid]:
                    if hour < days*24-1:
                        if not queue: raise RuntimeError(f'Patient queue exhausted: {role}, hour {hour}')
                        beds[bed] = (queue.pop(),0)
                else:
                    beds[bed] = (pid,pos)
    keys = pd.DataFrame(selected,columns=['patient_id','position','role','bed','hour'])
    # Per-patient position indexes the sorted, causally filled observation.
    lookup = PATIENT_TABLE.reindex(pd.MultiIndex.from_frame(keys[['patient_id','position']])).reset_index()
    schedule = pd.concat([keys[['role','bed','hour']],lookup],axis=1)
    schedule['day'] = schedule.hour // 24 + 1
    assert len(schedule)==days*24*200
    assert schedule.groupby('hour').size().eq(200).all()
    assert schedule.groupby(['hour','role']).size().eq(100).all()
    return schedule

### Pipeline selectivo, referencia de monitoreo y unidad de costo

El pipeline conserva transformación mediana/IQR con clipping y XGBoost. En v2
cada pipeline guarda además la **referencia de monitoreo** de los datos con los
que se ajustó su transformador: A2 y A4 la reemplazan, A1 y A3 la conservan.
Así KS/PSI miden el cambio desde la última actualización de features.

La unidad de costo se fija aquí, antes de calibrar o entrenar: mediana de cinco
ejecuciones de A4 sobre un día de pacientes de actualización de B-train.

In [ ]:
def build_reference(rows):
    """Freeze per-vital decile bins and a KS sample from the given rows."""
    reference = {}
    for col in VITALS:
        clean = rows[col].dropna().to_numpy(dtype=float)
        if len(clean) == 0:
            reference[col] = None
            continue
        edges = np.unique(np.quantile(clean, np.linspace(0, 1, 11)))
        edges = np.concatenate([[-np.inf], edges[1:-1], [np.inf]]) if len(edges) > 2 else np.array([-np.inf, np.inf])
        reference[col] = {'sample': clean[::max(1, len(clean)//1000)][:1000], 'edges': edges,
                          'hist': np.histogram(clean, bins=edges)[0]}
    return reference

class Pipeline:
    """Seven-vital transform and classifier with separately replaceable stages."""
    def __init__(self, med, iqr, model, ref=None):
        self.med = np.asarray(med,dtype=np.float32).copy()
        self.iqr = np.asarray(iqr,dtype=np.float32).copy()
        self.model = model
        self.ref = ref
    def transform(self, rows):
        """Apply current imputation, robust scaling and explicit clipping."""
        x=rows[VITALS].to_numpy(dtype=np.float32)
        x=np.where(np.isfinite(x),x,self.med)
        z=(x-self.med)/np.maximum(self.iqr,1e-6)
        return np.clip(z,-2,2)
    def predict(self, rows):
        """Return positive-class probability for observed rows."""
        return self.model.predict_proba(self.transform(rows))[:,1]
    def saturation(self, rows):
        """Fraction of feature cells at a clipping boundary."""
        x=rows[VITALS].to_numpy(dtype=np.float32)
        z=(np.where(np.isfinite(x),x,self.med)-self.med)/np.maximum(self.iqr,1e-6)
        return float(np.mean(np.abs(z)>2))
    def act(self, action, rows):
        """Return a new pipeline and measured action time without mutating self."""
        assert 0<=action<4
        start=time.perf_counter()
        if action==0:
            return self, time.perf_counter()-start
        med,iqr=self.med,self.iqr
        if action in (1,3):
            med=rows[VITALS].median().fillna(pd.Series(self.med,index=VITALS)).to_numpy(dtype=np.float32)
            q=rows[VITALS].quantile(.75)-rows[VITALS].quantile(.25)
            iqr=q.fillna(pd.Series(self.iqr,index=VITALS)).clip(lower=1e-6).to_numpy(dtype=np.float32)
        model=self.model
        if action in (2,3):
            if rows.y_dummy.nunique()<2: raise ValueError('Update needs both dummy classes')
            x=Pipeline(med,iqr,self.model).transform(rows)
            model=XGBClassifier(n_estimators=60,max_depth=3,learning_rate=.1,tree_method='hist',
                                device='cpu',n_jobs=2,random_state=SEED,eval_metric='logloss')
            model.fit(x,rows.y_dummy.to_numpy())
        elapsed=time.perf_counter()-start
        # Monitoring reference follows the transformer; not part of action cost.
        ref=build_reference(rows) if action in (1,3) else self.ref
        return Pipeline(med,iqr,model,ref),elapsed

initial_model=XGBClassifier(n_estimators=60,max_depth=3,learning_rate=.1,tree_method='hist',
                            device='cpu',n_jobs=2,random_state=SEED,eval_metric='logloss')
initial_model.fit(Pipeline(REF_MED,REF_IQR,None).transform(a_train),a_train.y_dummy.to_numpy())
INITIAL=Pipeline(REF_MED,REF_IQR,initial_model)
probe_rows=a_train.iloc[:2400]
assert probe_rows.y_dummy.nunique()==2
for probe_action in (0,1,2,3):
    probe_pipe,probe_seconds=INITIAL.act(probe_action,probe_rows)
    assert probe_seconds>=0
    assert (probe_pipe.model is INITIAL.model)==(probe_action in (0,1))
    if probe_action in (0,2):assert np.array_equal(probe_pipe.med,INITIAL.med) and probe_pipe.ref is INITIAL.ref
    else:assert probe_pipe.ref is not INITIAL.ref
del probe_pipe,probe_rows
def auc(rows,pipe):
    """Return NaN for a one-class evaluation day."""
    y=rows.y_dummy.to_numpy()
    return float(roc_auc_score(y,pipe.predict(rows))) if np.unique(y).size==2 else np.nan
held_a=frame.loc[frame.patient_id.isin(a_test_ids)]
a_test_auc=auc(held_a,INITIAL)

cost_rows=make_replay(B_PARTS['train'],days=1,seed=SEED)
cost_rows=cost_rows.loc[cost_rows.role.eq('update')]
ACTION_SECONDS={ACTIONS[a]:float(np.median([INITIAL.act(a,cost_rows)[1] for _ in range(5)])) for a in range(4)}
COST_UNIT=max(ACTION_SECONDS['A4'],1e-6)
def utility(auc_value,seconds):
    """Next-day AUC minus cost measured in frozen A4-time units."""
    return float(auc_value-COST_LAMBDA*seconds/COST_UNIT)
del cost_rows
checkpoint('initial_model', a_test_auc=a_test_auc, action_seconds_b_train=ACTION_SECONDS, cost_unit_s=COST_UNIT,
           action_cost_units={k:COST_LAMBDA*v/COST_UNIT for k,v in ACTION_SECONDS.items()})
print('A-test AUC dummy:',round(a_test_auc,4))
print('Costo en reward por acción:',{k:round(COST_LAMBDA*v/COST_UNIT,4) for k,v in ACTION_SECONDS.items()})

### Cambios estadísticos graduales sobre copias del replay

Severidad diaria desde el inicio:

`L(d) = rate · (d − inicio + 1)` si `d ≥ inicio`, y 0 antes.

- **Covariate:** `x' = m_ref + s(L) · (x − m_ref) + L · iqr_ref · dirección`, con
  `s(L) = 1 + 0,4 · min(1, L/3)`. `L` se mide en IQR y no tiene tope.
- **Concept:** la etiqueta usa pesos `w(u)` interpolados linealmente entre las
  anclas `W → W·HR⁻ → W·(HR,Resp)⁻ → W·Resp⁻`, con `u = L` limitado a 3; el umbral
  es la mediana de A-train del score con `w(u)` y se aplica la misma inversión del 5%.
- **Combined:** ambas rampas; la de concepto empieza `CONCEPT_DELAY` días después.

Donde: `rate` es la velocidad diaria, `m_ref` e `iqr_ref` son mediana e IQR de
A-train y `dirección` alterna signos para que el cambio afecte a los siete vitales.
Los faltantes se mantienen y la transformación opera solo sobre copias numéricas.

In [ ]:
def ramp(day, start, rate, cap=np.inf):
    """Return daily severity: zero before start, then rate per elapsed day."""
    day = np.asarray(day, dtype=float)
    return np.where(day >= start, np.minimum((day-start+1)*rate, cap), 0.)

def inject(schedule,scenario,cov_rate=.35,concept_rate=.15,onset=ONSET,delay=CONCEPT_DELAY):
    """Apply gradual numeric stress while preserving missingness and provenance."""
    assert scenario in SCENARIOS
    out=schedule.copy()
    days=np.arange(1,int(out.day.max())+1)
    zero=np.zeros(len(days))
    cov_by_day=ramp(days,onset,cov_rate) if scenario in ('covariate','combined') else zero
    start=onset+(delay if scenario=='combined' else 0)
    con_by_day=ramp(days,start,concept_rate,CONCEPT_MAX) if scenario in ('concept','combined') else zero
    direction=np.array([1,-1,1,1,-1,1,1],dtype=np.float32)
    for i,d in enumerate(days):
        mask=out.day.eq(d).to_numpy()
        level=float(cov_by_day[i])
        if level>0:
            scale=1+SCALE_GAIN*min(1.,level/3)
            src=out.loc[mask,VITALS].to_numpy(dtype=np.float32)
            out.loc[mask,VITALS]=REF_MED+scale*(src-REF_MED)+level*REF_IQR*direction
        u=float(con_by_day[i])
        if u>0:
            score=out.loc[mask,Z_COLS].to_numpy()@concept_weights(u)
            out.loc[mask,'y_dummy']=np.logical_xor(score>concept_threshold(u),out.loc[mask,'flip'].to_numpy()).astype(np.int8)
    out['scenario']=scenario
    out['cov_level']=cov_by_day[out.day.to_numpy()-1]
    out['concept_level']=con_by_day[out.day.to_numpy()-1]
    out['drift_active']=(out.cov_level>0)|(out.concept_level>0)
    return out

assert ramp(np.arange(1,9),5,.5).tolist()==[0,0,0,0,.5,1.,1.5,2.]
assert np.allclose(concept_weights(1.5),(ANCHORS[1]+ANCHORS[2])/2)

### KS, PSI y AUC antes de actuar, relativos a la última actualización de features

La referencia inicial proviene de cuatro días de B-train. Cuando A2 o A4
reajustan el transformador, la referencia pasa a ser el día usado para ese
ajuste. Así KS/PSI miden el desplazamiento acumulado desde la última actualización.

In [ ]:
reference_replay=make_replay(B_PARTS['train'],days=4,seed=SEED)
REFERENCE=build_reference(reference_replay)
INITIAL.ref=REFERENCE
del reference_replay
def psi(reference,current):
    """Population stability index on frozen reference bins."""
    edges=reference['edges']
    if len(edges)<2: return 0.0
    q=np.histogram(np.asarray(current,dtype=float),bins=edges)[0].astype(float)+1e-6
    p=reference['hist'].astype(float)+1e-6
    p/=p.sum();q/=q.sum()
    return float(np.sum((q-p)*np.log(q/p)))
def monitor(rows,pipe):
    """Return maximum per-vital KS/PSI against the pipeline reference and held-out AUC."""
    ks_values=[];psi_values=[]
    for col in VITALS:
        now=rows[col].dropna().to_numpy(dtype=float)
        ref=pipe.ref[col]
        if len(now)==0 or ref is None: continue
        ks_values.append(float(ks_2samp(ref['sample'],now,method='asymp').statistic))
        psi_values.append(psi(ref,now))
    eval_rows=rows.loc[rows.role.eq('eval')]
    return {'ks':max(ks_values,default=0.),'psi':max(psi_values,default=0.),
            'auc':auc(eval_rows,pipe),'prevalence':float(eval_rows.y_dummy.mean()),
            'saturation':pipe.saturation(rows)}
def encode_observation(metrics,last_auc=.5):
    """Encode three thresholded metrics and an AUC-validity bit."""
    value=metrics['auc'] if np.isfinite(metrics['auc']) else last_auc
    bins=[np.searchsorted([.10,.30],metrics['ks'],side='right'),
          np.searchsorted([.10,.25],metrics['psi'],side='right'),
          np.searchsorted([.65,.85],value,side='right')]
    result=np.zeros(10,dtype=np.float32)
    for i,b in enumerate(bins): result[i*3+b]=1
    result[9]=float(np.isfinite(metrics['auc']))
    return result

### Entorno con recompensa observable al cierre del día siguiente

La recompensa es `utilidad = AUC_día_siguiente − COST_LAMBDA · segundos / COST_UNIT`.

Donde:

- `AUC_día_siguiente`: AUC de pacientes de evaluación el día d+1 con el pipeline resultante.
- `segundos`: duración efectiva de la acción medida con `perf_counter`.
- `COST_UNIT`: mediana de segundos de A4 sobre B-train, fijada antes de entrenar.
- `COST_LAMBDA = 0,05`: AUC equivalente al costo de una ejecución típica de A4.

El costo deja de ser despreciable frente a variaciones diarias de AUC y ordena
las acciones por su trabajo real (A1 < A2 < A3 < A4). El entorno registra
además la severidad diaria de cada cambio, oculta para el agente.

In [ ]:
class BedEnv:
    """Partially observed daily maintenance environment over one replay."""
    def __init__(self,schedule,initial=INITIAL):
        self.schedule=schedule
        self.initial=initial
        self.days=int(schedule.day.max())
        self.parts={d:schedule.loc[schedule.day.eq(d)] for d in range(1,self.days+1)}
        self.reset()
    def regime(self,day):
        """Return hidden covariate and concept severities for one day."""
        rows=self.parts[day]
        return float(rows.cov_level.iloc[0]),float(rows.concept_level.iloc[0])
    def reset(self):
        """Restore initial pipeline and return the first three-day history."""
        self.pipe=self.initial
        self.day=1
        self.history=deque(maxlen=3)
        self.last_auc=.5
        self.actions=[];self.daily=[]
        self.current=self._observe(1)
        return self.current[0],{}
    def _observe(self,day):
        measured=monitor(self.parts[day],self.pipe)
        block=encode_observation(measured,self.last_auc)
        if np.isfinite(measured['auc']):self.last_auc=measured['auc']
        self.history.append(block)
        vector=np.concatenate([np.zeros(10,dtype=np.float32)]*(3-len(self.history))+list(self.history))
        return vector,measured
    def step(self,action):
        """Act at day close and reveal next-day held-out utility."""
        if self.day>=self.days:raise RuntimeError('Episode already finished')
        before=self.current[1]
        day=self.day
        update=self.parts[day].loc[self.parts[day].role.eq('update')]
        old=self.pipe
        try:
            changed,elapsed=old.act(int(action),update)
            executed=int(action);reason=''
        except ValueError as exc:
            changed,elapsed=old.act(0,update)
            executed=0;reason=str(exc)
        self.pipe=changed
        self.day+=1
        self.current=self._observe(self.day)
        next_state,after=self.current
        valid=bool(np.isfinite(after['auc']))
        reward=utility(after['auc'],elapsed) if valid else np.nan
        done=self.day==self.days
        (cov,con),(next_cov,next_con)=self.regime(day),self.regime(self.day)
        record={'day':day,'scenario':str(self.schedule.scenario.iloc[0]),'requested':int(action),
                'executed':executed,'fallback_reason':reason,'time_s':float(elapsed),
                'cost':float(COST_LAMBDA*elapsed/COST_UNIT),
                'auc_before':before['auc'],'auc_next':after['auc'],'reward':reward,
                'reward_valid':valid,'drift_active':bool(cov or con),'next_drift_active':bool(next_cov or next_con),
                'cov_level':cov,'concept_level':con,'next_cov_level':next_cov,'next_concept_level':next_con,
                'ks':before['ks'],'psi':before['psi'],'saturation':before['saturation'],
                'saturation_next':after['saturation']}
        self.actions.append(record)
        self.daily.append({'day':day,'scenario':record['scenario'],**before,'action':ACTIONS[executed],
                           'reward':reward,'time_s':float(elapsed),'drift_active':record['drift_active'],
                           'cov_level':cov,'concept_level':con})
        if done:
            self.daily.append({'day':self.day,'scenario':record['scenario'],**after,'action':'—',
                               'reward':np.nan,'time_s':0.,'drift_active':record['next_drift_active'],
                               'cov_level':next_cov,'concept_level':next_con})
        return next_state,reward,done,False,{'record':record,'reward_valid':valid,'next_metrics':after}

### Double DQN con replay FIFO, red target congelada y pérdida Huber

Se conservan arquitectura, replay, batch 64, gamma 0,99 y copia dura cada 50
actualizaciones. El objetivo de Bellman en v3 es:

`y = (r − 0,8) + γ · (1 − done) · Q_target(s', argmax_a Q_online(s', a))`

Donde:

- `r`: recompensa observada; `0,8` es el centrado constante de v1/v2.
- `γ = 0,99`: descuento; `done` anula el término en el último día.
- `Q_online`: red entrenada, que **elige** la acción siguiente.
- `Q_target`: copia congelada, que **evalúa** esa acción.

Separar elección y evaluación reduce la sobreestimación por ruido. La pérdida
Huber limita el efecto de errores grandes. La tasa de aprendizaje se fija al
final de cada episodio con interpolación lineal entre 5e-4 y 5e-5.

In [ ]:
class QNetwork(nn.Module):
    """Estimate four action values from the last three daily observations."""
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(30,128),nn.ReLU(),nn.Linear(128,128),nn.ReLU(),nn.Linear(128,4))
    def forward(self,x):
        """Return unconstrained Q values."""
        return self.net(x)

class ReplayBuffer:
    """FIFO transitions with uniform minibatch sampling."""
    def __init__(self,capacity=50000):
        self.buffer=deque(maxlen=capacity)
        self.rng=random.Random(SEED)
    def push(self,s,a,r,s2,done):
        """Store a complete observed transition."""
        self.buffer.append((s.copy(),int(a),float(r),s2.copy(),bool(done)))
    def sample(self,n):
        """Return a uniform minibatch."""
        batch=self.rng.sample(self.buffer,n)
        s,a,r,s2,d=zip(*batch)
        return (np.stack(s),np.array(a),np.array(r,dtype=np.float32),np.stack(s2),np.array(d,dtype=np.float32))
    def __len__(self):
        """Return current capacity use."""
        return len(self.buffer)

class DQNAgent:
    """Epsilon-greedy DQN with hard target copies every 50 updates.

    Bellman values center reward by a constant for numerical learning; all
    episodes have the same horizon, so this action-independent shift preserves
    action ordering. CSV and replay retain the specified uncentered reward.
    Epsilon and learning rate decay once per finished episode. Targets use
    Double DQN and the loss is Huber.
    """
    def __init__(self):
        self.online=QNetwork().to(DEVICE)
        self.target=QNetwork().to(DEVICE)
        self.target.load_state_dict(self.online.state_dict())
        self.target.requires_grad_(False)
        self.optim=torch.optim.Adam(self.online.parameters(),lr=LR_START)
        self.replay=ReplayBuffer()
        self.rng=np.random.default_rng(SEED)
        self.epsilon=1.
        self.updates=0
        self.losses=[]
    def q_values(self,obs):
        """Return online Q values for one observation."""
        with torch.no_grad():
            s=torch.as_tensor(obs,dtype=torch.float32,device=DEVICE).unsqueeze(0)
            return self.online(s).squeeze(0).cpu().numpy()
    def select(self,obs,greedy=False):
        """Choose random exploration or highest online Q value."""
        if not greedy and self.rng.random()<self.epsilon:return int(self.rng.integers(4))
        return int(np.argmax(self.q_values(obs)))
    def learn(self,s,a,r,s2,done):
        """Add valid delayed outcome and make one Bellman gradient step."""
        if not np.isfinite(r):return None
        self.replay.push(s,a,r,s2,done)
        if len(self.replay)<64:return None
        states,actions,rewards,next_states,terminals=self.replay.sample(64)
        states=torch.as_tensor(states,dtype=torch.float32,device=DEVICE)
        actions=torch.as_tensor(actions,dtype=torch.long,device=DEVICE).unsqueeze(1)
        rewards=torch.as_tensor(rewards,dtype=torch.float32,device=DEVICE)
        following=torch.as_tensor(next_states,dtype=torch.float32,device=DEVICE)
        terminal=torch.as_tensor(terminals,dtype=torch.float32,device=DEVICE)
        values=self.online(states).gather(1,actions).squeeze(1)
        with torch.no_grad():
            next_actions=self.online(following).argmax(1,keepdim=True)
            next_values=self.target(following).gather(1,next_actions).squeeze(1)
            targets=(rewards-.8)+.99*(1-terminal)*next_values
        loss=nn.functional.smooth_l1_loss(values,targets)
        self.optim.zero_grad();loss.backward()
        torch.nn.utils.clip_grad_value_(self.online.parameters(),10.)
        self.optim.step()
        self.updates+=1
        if self.updates%50==0:self.target.load_state_dict(self.online.state_dict())
        self.losses.append(float(loss.item()))
        return float(loss.item())
    def end_episode(self,progress_fraction):
        """Decay exploration and set linearly decayed learning rate."""
        self.epsilon=max(EPS_MIN,self.epsilon*EPS_DECAY)
        for group in self.optim.param_groups:
            group['lr']=LR_START+(LR_END-LR_START)*min(1.,progress_fraction)
    @property
    def lr(self):
        """Return current optimizer learning rate."""
        return float(self.optim.param_groups[0]['lr'])

### Contratos básicos antes del entrenamiento largo

Se comprueban tamaño de observación, capacidad FIFO, congelamiento de la red
target, decaimiento de epsilon y tasa de aprendizaje, y un paso Double DQN.

In [ ]:
assert len(encode_observation({'ks':0.,'psi':0.,'auc':.9}))==10
small_buffer=ReplayBuffer(2)
for i in range(3):small_buffer.push(np.zeros(30,dtype=np.float32),0,.5,np.zeros(30,dtype=np.float32),False)
assert len(small_buffer)==2
temp_agent=DQNAgent()
assert all(not p.requires_grad for p in temp_agent.target.parameters())
assert len({id(p) for g in temp_agent.optim.param_groups for p in g['params']} & {id(p) for p in temp_agent.target.parameters()})==0
temp_agent.end_episode(0.)
assert abs(temp_agent.epsilon-EPS_DECAY)<1e-12 and abs(temp_agent.lr-LR_START)<1e-12
temp_agent.end_episode(1.)
assert abs(temp_agent.lr-LR_END)<1e-12
for _ in range(64):temp_agent.learn(np.zeros(30,dtype=np.float32),1,.9,np.zeros(30,dtype=np.float32),True)
assert temp_agent.updates==1 and np.isfinite(temp_agent.losses[-1])
assert temp_agent.q_values(np.zeros(30,dtype=np.float32)).shape==(4,)
del temp_agent,small_buffer

### Calibrar velocidades solo con B-train y congelarlas antes de prueba

Para cada velocidad, con inicio en el día 5 y el pipeline inicial sin actualizar:

1. **Lento:** la caída de AUC el primer día de cambio debe ser < 0,05.
2. **Degradación fuerte:** tras seis días de cambio la caída acumulada debe ser > 0,15.
3. **Reparable:** actualizar ese sexto día con la acción adecuada (A2 para
   medición, A3 para concepto) debe recuperar > 0,10 de AUC al día siguiente.

Se elige la velocidad **más lenta** que cumple los tres criterios, por separado
para covariate y concept.

In [ ]:
def calibrate():
    """Select slowest covariate and concept rates with slow onset, strong decay and repair."""
    base=make_replay(B_PARTS['train'],days=max(ONSET+CALIBRATION_DAYS+1,DAYS),seed=SEED)
    def rows(schedule,day,role):return schedule.loc[schedule.day.eq(day)&schedule.role.eq(role)]
    first,late=ONSET,ONSET+CALIBRATION_DAYS-1
    stable_first=auc(rows(base,first,'eval'),INITIAL)
    stable_late=auc(rows(base,late,'eval'),INITIAL)
    records=[]
    grids=(('covariate',COV_RATE_GRID,1),('concept',CONCEPT_RATE_GRID,2))
    for kind,grid,action in grids:
        for rate in grid:
            kwargs={'cov_rate':rate} if kind=='covariate' else {'concept_rate':rate}
            shifted=inject(base,kind,onset=ONSET,**kwargs)
            first_auc=auc(rows(shifted,first,'eval'),INITIAL)
            late_auc=auc(rows(shifted,late,'eval'),INITIAL)
            repaired,_=INITIAL.act(action,rows(shifted,late,'update'))
            next_stale=auc(rows(shifted,late+1,'eval'),INITIAL)
            next_repaired=auc(rows(shifted,late+1,'eval'),repaired)
            records.append({'kind':kind,'rate':rate,'first_day_drop':stable_first-first_auc,
                            'late_drop':stable_late-late_auc,'late_auc':late_auc,
                            'repair_gain':next_repaired-next_stale,'repaired_auc':next_repaired})
    table=pd.DataFrame(records)
    table['qualified']=(table.first_day_drop<.05)&(table.late_drop>.15)&(table.repair_gain>.10)
    table.to_csv(OUT/'calibration.csv',index=False)
    chosen={}
    for kind,_,_ in grids:
        part=table.loc[table.kind.eq(kind)].sort_values('rate')
        good=part.loc[part.qualified]
        chosen[kind]=float((good if len(good) else part.sort_values('late_drop',ascending=False)).iloc[0].rate)
    return chosen['covariate'],chosen['concept'],bool(table.groupby('kind').qualified.any().all()),table

### Reproducir una política sin acceso al escenario durante decisión

Políticas: DQN greedy, A1–A4 permanentes, aleatoria y heurística por umbrales.
El oracle inmediato evalúa las cuatro acciones desde el mismo pipeline y el
mismo día siguiente; es diagnóstico, no política óptima de largo plazo. Una
decisión es **frontera** si el cambio empieza entre d y d+1 (inicio de rampa).

In [ ]:
def choose_policy(policy,agent,obs,metrics,rng):
    """Choose among DQN, four fixed actions, random and fixed thresholds."""
    if policy=='dqn':return agent.select(obs,greedy=True)
    if policy in ACTIONS:return ACTIONS.index(policy)
    if policy=='random':return int(rng.integers(4))
    if policy=='heuristic':
        cov=metrics['ks']>=.30 or metrics['psi']>=.25
        low=np.isfinite(metrics['auc']) and metrics['auc']<.85
        return 3 if cov and low else 1 if cov else 2 if low else 0
    raise ValueError(policy)

def roll(schedule,policy,agent=None,audit=False):
    """Return auditable action/day records and optional one-step action oracle."""
    env=BedEnv(schedule)
    obs,_=env.reset()
    rng=np.random.default_rng(SEED)
    audit_rows=[]
    while env.day<env.days:
        day=env.day
        metrics=env.current[1]
        action=choose_policy(policy,agent,obs,metrics,rng)
        q=agent.q_values(obs) if policy=='dqn' else np.full(4,np.nan)
        if audit:
            update=env.parts[day].loc[env.parts[day].role.eq('update')]
            next_eval=env.parts[day+1].loc[env.parts[day+1].role.eq('eval')]
            utilities=[]
            for candidate in range(4):
                try:
                    new,elapsed=env.pipe.act(candidate,update)
                    candidate_auc=auc(next_eval,new)
                    utilities.append(utility(candidate_auc,elapsed) if np.isfinite(candidate_auc) else np.nan)
                except ValueError:utilities.append(np.nan)
        obs,reward,done,_,info=env.step(action)
        info['record'].update({f'q_{ACTIONS[i]}':float(q[i]) for i in range(4)})
        if audit:
            chosen=info['record']
            utilities[chosen['executed']]=chosen['reward']
            best=float(np.nanmax(utilities)) if np.isfinite(utilities).any() else np.nan
            regime=(chosen['cov_level'],chosen['concept_level'])
            next_regime=(chosen['next_cov_level'],chosen['next_concept_level'])
            audit_rows.append({'scenario':chosen['scenario'],'day':day,'chosen':ACTIONS[chosen['executed']],
                               'oracle':ACTIONS[int(np.nanargmax(utilities))] if np.isfinite(best) else '',
                               'cov_level':regime[0],'concept_level':regime[1],
                               'next_cov_level':next_regime[0],'next_concept_level':next_regime[1],
                               'frontier':bool(chosen['drift_active']!=chosen['next_drift_active'])
                                          or bool((regime[0]==0)!=(next_regime[0]==0)) or bool((regime[1]==0)!=(next_regime[1]==0)),
                               **{f'utility_{ACTIONS[i]}':utilities[i] for i in range(4)},
                               'chosen_utility':chosen['reward'],'oracle_utility':best,
                               'regret':best-chosen['reward'] if np.isfinite(best) and np.isfinite(chosen['reward']) else np.nan})
        if done:break
    actions=pd.DataFrame(env.actions).assign(policy=policy)
    daily=pd.DataFrame(env.daily).assign(policy=policy)
    valid=actions.reward.dropna()
    result={'scenario':schedule.scenario.iloc[0],'policy':policy,
            'mean_auc':float(daily.auc.mean()),'return':float(valid.sum()),
            'mean_reward':float(valid.mean()) if len(valid) else np.nan,
            'seconds':float(actions.time_s.sum()),'cost':float(actions.cost.sum()),'n_valid':int(len(valid)),
            **{f'count_{name}':int(actions.executed.eq(i).sum()) for i,name in enumerate(ACTIONS)}}
    return result,actions,daily,pd.DataFrame(audit_rows)

### Entrenar DQN y elegir checkpoint solo con validación

Cada 10 episodios se evalúa la política greedy en validación. Además de los
criterios de v2, se guardan por checkpoint la coincidencia de acciones con el
checkpoint previo y la versión con tolerancia Q (diagnóstico). El checkpoint
final es el último de la ventana de tres validaciones con mayor media.

Los episodios siguen `TRAIN_CYCLE` (seis de cada siete con cambio). En cada
episodio de entrenamiento el inicio (días 3–7) y un factor de velocidad
(0,75; 1; 1,25 × velocidad calibrada) se sortean con un generador de semilla fija,
para que el agente no memorice un calendario ni una pendiente. Validación y
prueba usan inicio 5 y las velocidades calibradas.

In [ ]:
def agreement(previous,current,tolerance=None):
    """Share of validation decisions repeated; optional Q-value tie tolerance."""
    before=np.array(previous['signature']);after=np.array(current['signature'])
    same=before==after
    if tolerance is not None:
        q=current['q'];rows=np.arange(len(after))
        same|=(q[rows,after]-q[rows,before])<=tolerance
    return float(np.mean(same))

def train_agent(cov_rate,concept_rate):
    """Run drift-weighted scenario episodes and frozen validation checkpoints."""
    agent=DQNAgent()
    train_replays=[make_replay(B_PARTS['train'],days=DAYS,seed=SEED+i) for i in range(4)]
    val_base=make_replay(B_PARTS['val'],days=DAYS,seed=SEED)
    val_schedules=[inject(val_base,name,cov_rate,concept_rate) for name in SCENARIOS]
    calendar_rng=np.random.default_rng(SEED+1000)
    history=[];checkpoints=[]
    max_eps=MAX_EPISODES
    minimum=MIN_EPISODES
    for ep in range(1,max_eps+1):
        scenario=TRAIN_CYCLE[(ep-1)%len(TRAIN_CYCLE)]
        if DAYS>=8:
            onset=int(calendar_rng.integers(3,8));factor=float(calendar_rng.choice(TRAIN_RATE_FACTORS))
        else:
            onset,factor=ONSET,1.
        episode=inject(train_replays[(ep-1)%4],scenario,cov_rate*factor,concept_rate*factor,onset=onset)
        env=BedEnv(episode)
        obs,_=env.reset()
        episode_return=0.;counts=np.zeros(4,dtype=int)
        while env.day<env.days:
            action=agent.select(obs)
            next_obs,reward,done,_,info=env.step(action)
            counts[info['record']['executed']]+=1
            if info['reward_valid']:
                agent.learn(obs,action,reward,next_obs,done)
                episode_return+=reward
            obs=next_obs
            if done:break
        row={'episode':ep,'scenario':scenario,'onset':onset,'rate_factor':factor,'train_return':episode_return,
             'epsilon':agent.epsilon,'lr':agent.lr,'loss':agent.losses[-1] if agent.losses else np.nan,
             'updates':agent.updates,'validation_return':np.nan,
             **{f'train_count_{name}':int(counts[i]) for i,name in enumerate(ACTIONS)}}
        agent.end_episode(ep/max_eps)
        if ep%10==0 or ep==max_eps or (SMOKE and ep==minimum):
            vals=[];signatures=[];qs=[];val_counts=np.zeros(4,dtype=int)
            for val in val_schedules:
                result,actions,_,_=roll(val,'dqn',agent)
                vals.append(result['return']);signatures.extend(actions.executed.to_list())
                qs.append(actions[[f'q_{name}' for name in ACTIONS]].to_numpy())
                val_counts+=np.bincount(actions.executed,minlength=4)
            row['validation_return']=float(np.mean(vals))
            row.update({f'val_count_{name}':int(val_counts[i]) for i,name in enumerate(ACTIONS)})
            current={'episode':ep,'return':row['validation_return'],'weights':copy.deepcopy(agent.online.state_dict()),
                     'signature':signatures,'q':np.concatenate(qs)}
            if checkpoints:
                row['agreement_prev']=agreement(checkpoints[-1],current)
                row['tie_agreement_prev']=agreement(checkpoints[-1],current,Q_TIE)
            checkpoints.append(current)
            pd.DataFrame(history+[row]).to_csv(OUT/'training_history.csv',index=False)
            checkpoint('training',episodes_done=ep,updates=agent.updates,best_validation_return=max(c['return'] for c in checkpoints))
        history.append(row)
        if ep>=minimum and len(checkpoints)>=5:
            latest=checkpoints[-5:]
            scores=np.array([item['return'] for item in latest])
            spread=float(np.ptp(scores)/max(abs(scores.mean()),1e-6))
            same=float(np.mean(np.array(latest[-1]['signature'])==np.array(latest[-2]['signature'])))
            if spread<=.05 and same>=.95:break
    returns=pd.Series([c['return'] for c in checkpoints])
    smoothed=returns.rolling(min(SELECT_WINDOW,len(returns)),min_periods=1).mean()
    winner=checkpoints[int(smoothed.idxmax())]
    agent.online.load_state_dict(winner['weights'])
    pd.DataFrame(history).to_csv(OUT/'training_history.csv',index=False)
    spread=np.array([x['return'] for x in checkpoints[-5:]],dtype=float)
    variation=float(np.ptp(spread)/max(abs(spread.mean()),1e-6)) if len(spread)==5 else np.nan
    stability=agreement(checkpoints[-2],checkpoints[-1]) if len(checkpoints)>1 else np.nan
    tie_stability=agreement(checkpoints[-2],checkpoints[-1],Q_TIE) if len(checkpoints)>1 else np.nan
    return agent,history,checkpoints,{'selected_episode':winner['episode'],'selected_smoothed_return':float(smoothed.max()),
                                    'updates':agent.updates,'episodes':len(history),'validation_variation':variation,
                                    'policy_stability':stability,'policy_stability_q_tie':tie_stability,
                                    'final_lr':agent.lr,'final_epsilon':agent.epsilon}

### Nueve figuras de ocupación, cambio, entrenamiento, decisiones y AUC frente a no actuar

Las figuras de acciones y oracle marcan con líneas el inicio de cada rampa
(azul discontinua: medición; naranja punteada: relación features–etiqueta).

In [ ]:
COLORS=['#879ba5','#478da7','#c18d48','#a95a79']
def stage_starts(table,column):
    """Return the first day of a hidden ramp, if any."""
    active=table.loc[table[column].gt(0),'day']
    return [int(active.min())] if len(active) else []

def mark_stages(axis,table,ymin=None,ymax=None):
    """Draw covariate and concept ramp starts on an axis."""
    for column,style,color in (('cov_level','--','#245c78'),('concept_level',':','#bd8e49')):
        for d in stage_starts(table,column):
            if ymin is None:axis.axvline(d,ls=style,color=color,lw=1.2)
            else:axis.vlines(d,ymin,ymax,linestyles=style,colors=color,lw=1.5)

def make_figures(stable_schedule,stage_table,train_history,daily,actions,audit,comparison):
    """Save eight 16:9 PNG/SVG figures from observed, auditable tables."""
    plt.rcParams.update({'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
    files=[]
    def save(fig,name):
        fig.tight_layout()
        for ext in ('png','svg'):
            path=OUT/'figures'/f'{name}.{ext}'
            fig.savefig(path,dpi=160 if ext=='png' else None,bbox_inches='tight')
            files.append(str(path.relative_to(OUT)))
        plt.close(fig)
    fig,ax=plt.subplots(2,1,figsize=(16,9),sharex=True)
    occ=stable_schedule.groupby('hour').bed.nunique()
    ax[0].plot(occ.index/24+1,occ.values,color='#245c78')
    ax[0].set(ylabel='Camas ocupadas',ylim=(0,220),title='Ocupacion del replay: 200 camas cada hora')
    ordered=stable_schedule.sort_values(['bed','hour']).copy()
    ordered['replacement']=ordered.groupby('bed').patient_id.transform(lambda s:s.ne(s.shift()).astype(int))
    counts=ordered.groupby('day').replacement.sum()
    ax[1].bar(counts.index,counts.values,color='#55a39a')
    ax[1].set(xlabel='Dia simulado',ylabel='Pacientes ingresados por dia',title='Ingresos observados (dia 1 incluye 200 camas iniciales)')
    save(fig,'01_ocupacion_recambio')

    fig,axes=plt.subplots(1,2,figsize=(16,9),sharex=True)
    for axis,column,title,unit in ((axes[0],'cov_level','Severidad de medicion','Desplazamiento (IQR)'),
                                   (axes[1],'concept_level','Severidad de relacion features-etiqueta','Posicion en trayectoria (0 = original)')):
        for j,name in enumerate(SCENARIOS):
            table=stage_table.loc[stage_table.scenario.eq(name)].sort_values('day')
            axis.plot(table.day,table[column],marker='o',ms=4,lw=2,color=COLORS[j],ls='--' if name=='combined' else '-',label=name)
        axis.set(title=title,xlabel='Dia simulado',ylabel=unit)
        axis.legend()
    axes[1].axhline(1,ls=':',color='#9aa0a6');axes[1].text(1,1.02,'ancla 1: HR invertido',fontsize=9,color='#5f6670')
    fig.suptitle('Calendario de prueba: severidad diaria creciente (oculta al agente)')
    save(fig,'02_calendario_cambios')

    chosen=daily.loc[daily.policy.eq('dqn') & daily.scenario.eq('combined')].sort_values('day')
    fig,axes=plt.subplots(3,1,figsize=(16,9),sharex=True)
    for axis,col,thr in zip(axes,('ks','psi','auc'),(.30,.25,.85)):
        axis.plot(chosen.day,chosen[col],marker='o',color='#245c78',label=col.upper())
        axis.axhline(thr,ls='--',color='#a43b3b',label=f'Umbral {thr:.2f}')
        mark_stages(axis,chosen)
        if col=='psi':axis.set_yscale('symlog',linthresh=.25)
        axis.set_ylabel(col.upper());axis.legend(loc='best')
    axes[0].set_title('Monitoreo diario antes de accion (DQN, combinado); referencia = ultima actualizacion de features')
    axes[-1].set_xlabel('Dia simulado')
    save(fig,'03_monitoreo_ks_psi_auc')

    hist=pd.DataFrame(train_history)
    fig,axes=plt.subplots(4,1,figsize=(16,9),sharex=True)
    axes[0].plot(hist.episode,hist.train_return,alpha=.3,label='Retorno entrenamiento crudo')
    axes[0].plot(hist.episode,hist.train_return.rolling(min(20,len(hist)),min_periods=1).mean(),label='Media movil 20')
    valid=hist.dropna(subset=['validation_return'])
    axes[0].plot(valid.episode,valid.validation_return,'o-',label='Validacion greedy')
    axes[0].set(ylabel='Retorno episodico',title='Aprendizaje del DQN; validacion sin exploracion')
    axes[0].legend()
    axes[1].plot(hist.episode,hist.epsilon,color='#bd8e49',label='Epsilon')
    axes[1].plot(hist.episode,hist.lr/LR_START,color='#245c78',label='LR / LR inicial')
    axes[1].set(ylabel='Escala');axes[1].legend(loc='upper right')
    if 'agreement_prev' in valid:
        axes[2].plot(valid.episode,valid.agreement_prev,'o-',ms=3,color='#a95a79',label='Coincidencia exacta')
        axes[2].plot(valid.episode,valid.tie_agreement_prev,'o-',ms=3,color='#55a39a',label=f'Con tolerancia Q {Q_TIE}')
        axes[2].axhline(.95,ls='--',color='#a43b3b',label='Umbral 0,95')
        axes[2].set(ylabel='Acuerdo con\ncheckpoint previo',ylim=(0,1.05));axes[2].legend(ncol=3,loc='lower right')
    if 'val_count_A1' in valid:
        bottom=np.zeros(len(valid))
        for i,name in enumerate(ACTIONS):
            axes[3].bar(valid.episode,valid[f'val_count_{name}'],bottom=bottom,width=6,color=COLORS[i],label=name)
            bottom+=valid[f'val_count_{name}'].to_numpy()
        axes[3].legend(ncol=4,title='Acciones greedy en validacion',loc='lower right')
    axes[3].set(xlabel='Episodio',ylabel='Decisiones')
    save(fig,'04_entrenamiento_retorno_epsilon')

    fig,ax=plt.subplots(figsize=(16,9))
    ax.plot(hist.episode,hist.train_return,alpha=.3,label='Retorno entrenamiento crudo')
    ax.plot(hist.episode,hist.train_return.rolling(min(100,len(hist)),min_periods=1).mean(),lw=2,label='Media movil 100')
    ax.set(xlabel='Episodio',ylabel='Retorno episodico',title='Retorno de entrenamiento del DQN')
    ax.legend()
    save(fig,'04a_retorno_entrenamiento')

    if 'val_count_A1' in valid:
        fig,ax=plt.subplots(figsize=(16,9))
        bottom=np.zeros(len(valid))
        for i,name in enumerate(ACTIONS):
            ax.bar(valid.episode,valid[f'val_count_{name}'],bottom=bottom,width=6,color=COLORS[i],label=name)
            bottom+=valid[f'val_count_{name}'].to_numpy()
        ax.set(xlabel='Episodio',ylabel='Decisiones en validacion (de 60)',
               title='Acciones greedy en validacion cada 10 episodios')
        ax.legend(ncol=4,title='Accion',loc='upper center',bbox_to_anchor=(.5,-.08))
        save(fig,'04b_acciones_validacion')

    plot_daily=daily.loc[daily.scenario.eq('combined')].sort_values('day')
    policies=('dqn','A1','A2','A3','A4','random','heuristic')
    fig,ax=plt.subplots(figsize=(16,9))
    for policy in policies:
        series=plot_daily.loc[plot_daily.policy.eq(policy)]
        ax.plot(series.day,series.auc,'-' if policy=='dqn' else '--',marker='.',label=policy,lw=2.5 if policy=='dqn' else 1.2)
    mark_stages(ax,plot_daily.loc[plot_daily.policy.eq('dqn')])
    ax.set(xlabel='Dia simulado',ylabel='AUC dia actual',title='AUC sobre pacientes reservados: escenario combinado')
    ax.legend(ncol=7,loc='lower left')
    save(fig,'05a_auc_politicas')

    fig,ax=plt.subplots(figsize=(16,9))
    for policy in policies:
        series=plot_daily.loc[plot_daily.policy.eq(policy)]
        ax.plot(series.day,series.reward.fillna(0).cumsum(),'-' if policy=='dqn' else '--',marker='.',label=policy,lw=2.5 if policy=='dqn' else 1.2)
    mark_stages(ax,plot_daily.loc[plot_daily.policy.eq('dqn')])
    ax.set(xlabel='Dia simulado',ylabel='Reward acumulado (AUC - costo)',
           title='Retorno acumulado: mismo replay para todas las politicas (combinado)')
    ax.legend(ncol=7,loc='upper left')
    save(fig,'05b_reward_acumulado_politicas')

    fig,ax=plt.subplots(figsize=(16,9))
    for j,name in enumerate(SCENARIOS):
        data=actions.loc[actions.policy.eq('dqn') & actions.scenario.eq(name)].sort_values('day')
        ax.scatter(data.day, np.full(len(data),j),c=[COLORS[int(a)] for a in data.executed],s=130,marker='s')
        oracle=audit.loc[audit.scenario.eq(name)].sort_values('day')
        ax.scatter(oracle.day,np.full(len(oracle),j+.25),c=[COLORS[ACTIONS.index(a)] for a in oracle.oracle],s=40,marker='v')
        mark_stages(ax,data,j-.4,j+.4)
    for i,name in enumerate(ACTIONS):ax.scatter([],[],c=COLORS[i],label=name,s=100,marker='s')
    ax.scatter([],[],c='grey',marker='v',s=40,label='Oracle inmediato')
    ax.set(xlabel='Dia de decision',yticks=range(4),yticklabels=SCENARIOS,title='Acciones DQN (cuadrados) y oracle inmediato (triangulos) en prueba reservada',xlim=(.5,DAYS))
    ax.legend(title='Accion',ncol=5,loc='upper left')
    save(fig,'06_acciones_dqn')

    subset=audit.loc[audit.scenario.eq('combined')].sort_values('day')
    fig,ax=plt.subplots(figsize=(16,9))
    for i,name in enumerate(ACTIONS):ax.plot(subset.day,subset[f'utility_{name}'],marker='.',label=name,color=COLORS[i])
    ax.plot(subset.day,subset.chosen_utility,'ko',ms=5,label='Elegida')
    mark_stages(ax,subset)
    ax.set(xlabel='Dia de decision',ylabel='AUC siguiente - costo',
           title='Oracle inmediato: cuatro acciones desde mismo pipeline (combinado)')
    ax.legend(ncol=5)
    save(fig,'07a_utilidad_acciones')

    fig,ax=plt.subplots(figsize=(16,9))
    ax.bar(subset.day,subset.regret,color='#a95a79')
    mark_stages(ax,subset)
    ax.set(xlabel='Dia de decision',ylabel='Regret inmediato',
           title='Diferencia frente a mejor utilidad del dia (combinado)')
    save(fig,'07b_regret')

    fig,axes=plt.subplots(2,1,figsize=(16,9),sharex=True)
    for name in ('covariate','combined'):
        data=daily.loc[daily.policy.eq('dqn') & daily.scenario.eq(name)].sort_values('day')
        axes[0].plot(data.day,data.saturation,marker='o',label=name)
        axes[1].plot(data.day,data.auc,marker='o',label=name)
    mark_stages(axes[0],daily.loc[daily.policy.eq('dqn') & daily.scenario.eq('covariate')])
    axes[0].set(ylabel='Fraccion de features saturados',title='Clipping bajo cambio de medicion gradual')
    axes[1].set(xlabel='Dia simulado',ylabel='AUC',title='Recuperacion predictiva observada')
    axes[0].legend()
    save(fig,'08_saturacion_y_recuperacion')

    fig,axes=plt.subplots(2,2,figsize=(16,9),sharex=True,sharey=True)
    line_style={'dqn':('#1f6f9e','-','DQN'),'A1':('#b3741f','--','A1 (sin acciones)')}
    for axis,name in zip(axes.flat,SCENARIOS):
        subset=daily.loc[daily.scenario.eq(name)]
        mark_stages(axis,subset.loc[subset.policy.eq('dqn')])
        axis.axhline(.85,color='#9aa0a6',ls=':',lw=1)
        for policy,(color,style,label) in line_style.items():
            series=subset.loc[subset.policy.eq(policy)].sort_values('day')
            axis.plot(series.day,series.auc,ls=style,color=color,lw=2,marker='o',ms=4,label=label)
        means=subset.groupby('policy').auc.mean()
        axis.set_title(f"{name}  ·  AUC media DQN {means['dqn']:.3f} vs A1 {means['A1']:.3f}",fontsize=11)
        axis.grid(axis='y',color='#eceef0',lw=.8)
    for axis in axes[1]:axis.set_xlabel('Dia simulado')
    for axis in axes[:,0]:axis.set_ylabel('AUC dia actual')
    handles,labels=axes[0,0].get_legend_handles_labels()
    fig.legend(handles,labels,loc='upper center',ncol=2,frameon=False,bbox_to_anchor=(.5,1.0))
    fig.suptitle('AUC diaria en prueba: DQN frente a no actuar (A1)',y=1.05)
    save(fig,'09_auc_dqn_vs_a1')
    return files

### Ejecución, persistencia parcial y comprobación honesta de criterios

Se calibra, entrena, compara contra baselines en B-test, audita cada decisión
y se evalúan los mismos gates de v2 y v3. La estabilidad con tolerancia Q se reporta
en `run_summary.json` como diagnóstico y no altera `validation_converged`.
Si un gate falla, el estado queda en `needs_improvement`.

In [ ]:
def run_experiment():
    """Calibrate, train, compare policies, audit decisions and render figures."""
    cov_rate,concept_rate,calibrated,grid=calibrate()
    checkpoint('calibrated',cov_rate_iqr_per_day=cov_rate,concept_rate_per_day=concept_rate,calibration_qualified=calibrated)
    print('Calibración B-train: covariate',cov_rate,'IQR/día | concept',concept_rate,'/día | criterio:',calibrated)
    agent,history,checkpoints,training=train_agent(cov_rate,concept_rate)
    checkpoint('trained',**training)
    test_base=make_replay(B_PARTS['test'],days=DAYS,seed=SEED)
    summary_rows=[];all_actions=[];all_daily=[];all_audit=[];stage_rows=[]
    for scenario in SCENARIOS:
        schedule=inject(test_base,scenario,cov_rate,concept_rate)
        stage_rows.append(schedule.groupby('day')[['cov_level','concept_level']].first().reset_index().assign(scenario=scenario))
        for policy in ('dqn',*ACTIONS,'random','heuristic'):
            result,acts,days,aud=roll(schedule,policy,agent,audit=(policy=='dqn'))
            summary_rows.append(result);all_actions.append(acts);all_daily.append(days)
            if len(aud):all_audit.append(aud)
        pd.DataFrame(summary_rows).to_csv(OUT/'policy_comparison.csv',index=False)
        checkpoint('evaluating',completed_scenarios=scenario)
    comparison=pd.DataFrame(summary_rows)
    actions=pd.concat(all_actions,ignore_index=True)
    daily=pd.concat(all_daily,ignore_index=True)
    audit=pd.concat(all_audit,ignore_index=True)
    stage_table=pd.concat(stage_rows,ignore_index=True)
    actions.to_csv(OUT/'actions.csv',index=False)
    daily.to_csv(OUT/'daily_metrics.csv',index=False)
    audit.to_csv(OUT/'action_audit.csv',index=False)
    stage_table.to_csv(OUT/'stage_calendar.csv',index=False)
    checkpoint('plotting')
    files=make_figures(inject(test_base,'stable',cov_rate,concept_rate),stage_table,history,daily,actions,audit,comparison)
    dqn=comparison.loc[comparison.policy.eq('dqn')].set_index('scenario')
    a1=comparison.loc[comparison.policy.eq('A1')].set_index('scenario')
    random_policy=comparison.loc[comparison.policy.eq('random')].set_index('scenario')
    inner=audit.loc[~audit.frontier & audit.regret.notna()]
    dqn_actions=actions.loc[actions.policy.eq('dqn'),'executed']
    action_set=set(dqn_actions)
    gates={'all_four_actions_functional':set(actions.executed)=={0,1,2,3},
           'positive_dqn_return':bool((dqn['return']>0).all()),
           'dqn_above_a1_random_drift':bool(all(dqn.loc[s,'return']>max(a1.loc[s,'return'],random_policy.loc[s,'return']) for s in SCENARIOS[1:])),
           'oracle_near_rate_80pct':bool(len(inner)>0 and inner.regret.le(.01).mean()>=.8),
           'dqn_used_a2_and_a3':{1,2}<=action_set,
           'dqn_selected_four_actions':action_set=={0,1,2,3},
           'validation_converged':bool(np.isfinite(training['validation_variation']) and training['validation_variation']<=.05 and training['policy_stability']>=.95),
           'calibration_qualified':calibrated}
    checkpoint('done',status='complete' if all(gates.values()) else 'needs_improvement',gates=gates,
               dqn_action_counts={ACTIONS[i]:int(dqn_actions.eq(i).sum()) for i in range(4)},
               oracle_action_counts=audit.oracle.value_counts().to_dict(),
               dqn_oracle_match=float((audit.chosen==audit.oracle).mean()),
               oracle_near_rate=float(inner.regret.le(.01).mean()) if len(inner) else None,
               plots=files,files=['training_history.csv','daily_metrics.csv','actions.csv','action_audit.csv',
                                  'policy_comparison.csv','calibration.csv','stage_calendar.csv'],
               training=training)
    print('Estado:',progress['status'],'| gates:',gates)
    print('Acciones DQN:',progress['dqn_action_counts'],'| oracle:',progress['oracle_action_counts'])
    print(comparison[['scenario','policy','mean_auc','return','cost']].round(3).to_string(index=False))

try:
    run_experiment()
except Exception as exc:
    checkpoint('failed',status='failed',error=repr(exc),traceback=traceback.format_exc())
    print('Experimento falló:',repr(exc))